# Workshop NordNET: Open-TYNDP Installation, Workflows and Exploring Outcomes

:::{note} At the end of this notebook, you will be able to:

**1. Install Open-TYNDP locally**
  - Learn how to install Open-TYNDP either using `git` or from the Windows Installer
  - Set up the model environment with pixi
  
**2. Run simple Open-TYNDP workflows**
  - Run a first Open-TYNDP workflow
  - Learn how to modify assumptions to test alternative scenarios

**3. Explore latest Open-TYNDP outcomes**
  - Investigate the latest model outcomes using PyPSA-Explorer
  - Benchmark Open-TYNDP scenario outcomes by accessing directly via the PyPSA.statistics API
:::

:::{warning}
This notebook uses pre-downloaded results and runs simplified workflows designed for learning purposes. It is not a substitute for running the full Open-TYNDP workflow. All examples and results are based on Open-TYNDP v0.7.1 — outputs may differ in other versions.
:::


:::{note}
If you have not set up Python on your computer, you can execute this tutorial in your browser via [Google Colab](https://colab.research.google.com/). Click the download button in the top right corner to download the `.ipynb` file and import it in [Google Colab](https://colab.research.google.com/).

Then install the required packages by uncommenting the cell below.

:::

## Preparation and Intro

In [ ]:
# uncomment for running this notebook on Colab
# !pip install packaging==25.0 --q
# !pip install pypsa pypsa-explorer pandas matplotlib numpy pdf2image cartopy
# !apt-get install poppler-utils -qq

In [ ]:
# Standard library imports
import os
import shutil
import zipfile
from datetime import datetime
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import pandas as pd
import pypsa
from pypsa_explorer import create_app

# Plot settings
pypsa.set_option("params.statistics.nice_names", True)
pypsa.set_option("params.statistics.drop_zero", True)
pypsa.set_option("params.statistics.round", 3)
plt.rcParams["figure.figsize"] = [14, 7]
clip = 1  # TWh
port = 8050

# Detect if running on Google Colab
try:
    from google.colab import output

    IN_COLAB = True
    print(f"This notebook is running on Google Colab!")
except ImportError:
    IN_COLAB = False
    print(f"This notebook is running locally !")

We use a custom unzip function here to preserve the original file timestamps. This matters for the workflow manager `Snakemake`, which uses timestamps to decide which files need to be re-run.


In [ ]:
def unzip_with_timestamps(zip_path, extract_to, keep_zip=True):
    """Unzip a file while preserving original file timestamps."""
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        for member in zip_ref.infolist():
            # Extract the file
            zip_ref.extract(member, extract_to)

            # Get the extracted file path
            extracted_path = os.path.join(extract_to, member.filename)

            # Get the modification time from the zip file
            date_time = datetime(*member.date_time)
            timestamp = date_time.timestamp()

            # Set both access and modification times
            os.utime(extracted_path, (timestamp, timestamp))
    if not keep_zip:
        os.remove(zip_path)

We'll download the latest Open-TYNDP results (v0.7.1) and some helper scripts. The results are pre-computed so you don't need to run the full optimisation yourself.

In [ ]:
urls = {
    "data/results-0.7.1.zip": "https://storage.googleapis.com/open-tyndp-data-store/outcomes/0.7.1/results-0.7.1.zip",
    "scripts/_helpers.py": "https://raw.githubusercontent.com/open-energy-transition/open-tyndp-workshops/refs/heads/main/open-tyndp-workshops/scripts/_helpers.py",
}

if os.path.basename(os.getcwd()) in ["open-tyndp-workshops", "content"]:
    os.makedirs("data", exist_ok=True)
    os.makedirs("scripts", exist_ok=True)
    for name, url in urls.items():
        if os.path.exists(name):
            print(f"File {name} already exists. Skipping download.")
        else:
            print(f"Retrieving {name} from storage.")
            urlretrieve(url, name)
            print(f"File available in {name}.")

    to_dir = "data/results-0.7.1"
    if not os.path.exists(to_dir):
        print(f"Unzipping data/results-0.7.1.zip.")
        unzip_with_timestamps("data/results-0.7.1.zip", "data/results-0.7.1")
    print(f"Latest NT results for Open-TYNDP v0.7.1 are available in '{to_dir}'.")

    print("Done")
else:
    print("Not in open-tyndp-workshops directory.")

And we'll also import some handy functions from the just retrieved `_helpers` script.

In [ ]:
from scripts._helpers import (
    display_code_lines,
    run_pypsa_explorer_in_colab,
    show_benchmarks,
)

## Workflow management using `Snakemake` and `pixi`

The Open-TYNDP workflow involves many interconnected steps: from retrieving raw input data, to processing this data and preparing the sector-coupled PyPSA networks, to optimizing the networks, to postprocessing and benchmarking the results.

To manage this complexity, Open-TYNDP uses two complementary tools:

1. **`Snakemake`** - A workflow management system that automatically figures out which analysis steps to run and in what order
2. **`pixi`** - A package manager that simplifies environment setup and provides easy-to-use shortcuts for running workflows

The combination of `Snakemake` and `pixi` allow Open-TYNDP to run with the flexibility to easily change configurations and run different scenarios.

### Reminder: `Snakemake`

<img src="https://raw.githubusercontent.com/open-energy-transition/open-tyndp-workshops/792b8474ab5096e5ab8db2822af4fcd9fe659eb6/open-tyndp-workshops/snakemake_logo.png" width="500px" />

The `Snakemake` workflow management system is a tool to create reproducible and scalable data analyses.
Workflows are described via a human readable, Python based language. They can be seamlessly scaled to server, cluster, grid, and cloud environments, without the need to modify the workflow definition.

Snakemake follows the [GNU Make](https://www.gnu.org/software/make) paradigm: workflows are defined in terms of so-called `rules` that specify how to create a set of output files from a set of input files. Dependencies between the rules are determined automatically, creating a DAG (directed acyclic graph) of jobs that can be automatically parallelized.

**Why does Open-TYNDP use Snakemake?**

Running the full TYNDP analysis involves many steps that depend on each other.

Snakemake can automatically:

- Determine which steps need to run based on what files already exist
- Figure out the correct order to run them
- Skip steps that don't need to be re-run
- Can run independent steps in parallel to save time

:::{note}

Snakemake documentation: https://snakemake.readthedocs.io/

Snakemake introduction from the [Open-TYNDP Workshop Series #2](https://open-energy-transition.github.io/open-tyndp-workshops/20250624-workshop-pypsa-02.html#the-snakemake-tool)

:::

#### Using `snakemake`

Snakemake workflows can be triggered in different ways:

1. **By target file**: specify the final output you want (using `results/my_output.nc` as an example output file name)
    ```bash
    snakemake -call results/my_output.nc
    ```

2. **By rule name**: call a specific step in the workflow (using `build_data` as an example rule/step name)

    ```bash
    snakemake -call build_data
    ```

    **NOTE:** You cannot call a rule that includes a wildcard without specifying what the wildcard should be filled with. Otherwise, Snakemake will not know what to propagate back.

3. **By entire workflow**: Use the common rule `all` to execute the entire workflow. It takes the final workflow output as its input and thus requires all previous dependent rules to be run as well
    ```bash
    snakemake -call all
    ```

##### The dry-run flag (`-n`)

A very important feature is the `-n` flag which executes a `dry-run`. It is recommended to always first execute a `dry-run` before the actual execution of a workflow. This simply prints out the DAG of the workflow to investigate without actually executing it.

```bash
! snakemake -call -n
```

### Introducing: `pixi`

<img src="https://raw.githubusercontent.com/prefix-dev/pixi/9f94b8a24353ca88d21846b72438c6066e8adc74/docs/assets/pixi-logo.svg" width="150px" />



`pixi` is a cross-platform, multi-language package management and workflow tool. It is built on the foundation of the `conda` ecosystem.

**Why does Open-TYNDP use `pixi`?**

`pixi` serves two important roles in the Open-TYNDP project:

**1. Environment Management**  
`pixi` automatically installs all the required Python packages and their correct versions, similar to `conda` but faster and more reliable. Pixi helps us not have to worry about package conflicts or missing dependencies.

**2. Simplified Commands**  
`pixi` also allows us to create shortcuts for long snakemake commands.

You can see the full Snakemake commands that pixi runs by looking in the `pixi.toml` file in the Open-TYNDP repository. For example, we have defined a shortcut for running the full Scenario Building workflow, called ``tyndp``:

```toml
tyndp-sb = """
    snakemake -call --configfile config/config.tyndp.yaml
"""
```

:::{note}

`pixi` documentation: https://pixi.prefix.dev/latest/.

:::


#### Using `pixi`

**Without pixi (raw Snakemake), you would run this command to run the full Scenario Building workflow:**
```bash
snakemake -call --configfile config/config.tyndp.yaml
```

**Using `pixi`, you just need to run:**
```bash
pixi run tyndp
```

Later on in this notebook, you will see that we are using `pixi` commands to keep things a little simpler.


# 1. Installing Open-TYNDP

There are multiple different alternatives for your to install the Open-TYNDP.

Locally:

1. Clone the Open-TYNDP repository using a terminal
2. Clone the Open-TYNDP repository by executing this notebook locally or on Colab
3. Use the Open-TYNDP Windows Installer (only compatible with Windows OS)

You can follow the links to navigate to the corresponding section below.

### Option 1: Clone via terminal

Open a terminal, navigate to your preferred folder, and run:

```bash
git clone https://github.com/open-energy-transition/open-tyndp.git
cd open-tyndp
pixi install -e open-tyndp
```


### Option 2: Clone via this notebook (locally or Collab)

If you are running this notebook locally and prefer not to use a terminal, the cells below will clone the repository and set up the environment for you automatically.


First, navigate into the `data/` folder:

In [ ]:
os.chdir("data/")
print("Directory changed to:", os.getcwd())

Clone the Open-TYNDP repository directly:

In [ ]:
%%capture
if os.path.basename(os.getcwd()) == "data" and not os.path.exists("open-tyndp"):
    ! git clone https://github.com/open-energy-transition/open-tyndp.git

In [ ]:
# Check open-tyndp was cloned successfully
if os.path.exists("open-tyndp"):
    print("Successfully cloned Open-TYNDP repository into local data folder!")

Now, navigate into the the `open-tyndp` directory:

In [ ]:
os.chdir("open-tyndp/")
print("Directory changed to:", os.getcwd())

In [ ]:
# Uncomment the next line for running this notebook on Colab or if you do not have pixi installed on your machine locally
# ! wget -qO- https://pixi.sh/install.sh | sh

:::{Note}

If pixi was installed successfully but your shell still can't find it, you may need to add it to your PATH manually. Use the following command to do so.

:::

In [ ]:
os.environ["PATH"] = os.path.expanduser("~/.pixi/bin") + ":" + os.environ["PATH"]

We can now use `pixi` to install the `open-tyndp` environment:

In [ ]:
! pixi install -e open-tyndp

And that's it: We are good to go!

### Option 3: The Windows-Installer

If you're on Windows, you can skip the manual setup entirely. We've put together a signed and verified installer that takes care of everything for you under the hood: 

- cloning the repository
- installing Pixi, 
- configuring the open-tyndp environment, 
- and setting up a PowerShell session that's ready to go.

Just follow [this short installation video](https://www.youtube.com/watch?v=krxRaeN3N-4) and you'll be set up in no time.

```{raw} html
<iframe width="560" height="315" 
src="https://www.youtube.com/embed/krxRaeN3N-4?si=IWkrv3r71JNh8Xg2" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>
```

# 2. Run the Open-TYNDP workflow

Now that we've successfully installed the Open-TYNDP repository and initialized the python environment, let's learn how to run an actual modelling workflow!

## High-level workflow

Before diving into the details of the Scenario Building (SB) workflow, it helps to understand the high-level logic behind the large set of rules that make the entire Open-TYNDP workflow.



In the following figure, we've illustrated a much simplified workflow that represents the high-level steps of Open-TYNDP.

<center>

![](https://raw.githubusercontent.com/open-energy-transition/open-tyndp-workshops/refs/heads/main/open-tyndp-workshops/minimal_workflow.png)

</center>

In order to see the actual Open-TYNDP Scenario Building workflow, we can execute a dry-run (`-n`) with our pixi task to execute the Scenario Building workflow

In [ ]:
! pixi run tyndp-sb --config run='{"name":"NT"}' -n

Visualized in a graph, this workflow looks like this:

<center>

![]()
![](https://raw.githubusercontent.com/open-energy-transition/open-tyndp-workshops/refs/heads/feat/trondheim-workshop/open-tyndp-workshops/dag_open_tyndp.png)

</center>

As you can see, this workflow is much more complex than our minimal example from before.

Nevertheless, the general idea remains the same. We retrieve data which we consequently process, then we prepare the model network and we solve it before we postprocess the results (summary, plotting, benchmarks).

We've filtered this down just a little bit, to make this a little easier to read and to focus on the workflow beginning with the preparation of the sector-coupled network:

<center>

![](https://raw.githubusercontent.com/open-energy-transition/open-tyndp-workshops/refs/heads/feat/trondheim-workshop/open-tyndp-workshops/dag_open_tyndp_filtered.png)

</center>

## Configuration Options

The Open-TYNDP workflow's settings are housed in `config/config.tyndp.yaml`. Feel free to open the file to explore the full extent of configuration settings we have available.

One key parameter is the run name, which sets which scenario(s) to run. The TYNDP 2024 includes three scenarios, each representing a different vision of how Europe's energy system could develop by 2030 and 2040:

1. **National Trends (NT)** — a baseline scenario based on current national energy and climate plans. It reflects what is likely to happen given existing policies and commitments, without assuming additional ambition beyond what is already legislated.
2. **Distributed Energy (DE)** — a scenario in which the energy transition is driven by decentralised solutions: prosumers, local renewables, community energy, and distributed storage play a central role.
3. **Global Ambition (GA)** — a scenario with higher climate ambition and a more centralised approach, driven by large-scale renewable deployment, interconnection, and cross-border energy trade.
   
By default, Open-TYNDP runs all three scenarios. To run a single scenario, change the `run: name:` parameter in `config/config.tyndp.yaml`, for example, set it to "NT" to run just the NT scenario:




In [ ]:
display_code_lines("config/config.tyndp.yaml", "yaml", 6, 8)

Some TYNDP specific parameters are for example the list of TYNDP renewable carriers (technologies) included in the modelling scope.

In [ ]:
display_code_lines("config/config.tyndp.yaml", "yaml", 54, 71)

You can modify the config settings within Open-TYNDP in two ways:
1. **Edit the config file directly**
2. **Override via command line** -- For example, by adding the following to your command: `--config 'run={"name":"NT"}' 'scenario={"planning_horizons":[2030]}'`

:::{tip}
In practice, editing the YAML configuration files directly in a text editor or IDE is much easier than using command-line overrides. We're using command-line options in this notebook for demonstration purposes, but for real life, we recommend modifying the config files directly!
:::

In [ ]:
! pixi run tyndp-sb --config 'run={"name":"NT"}' 'scenario={"planning_horizons":[2030]}' -n

## Task: Configure the settings in config files

So far, we've been setting the run settings via the command line. However, as we've mentioned, in real life it's much easier to go into the config files and edit them directly. So let's do that for the settings we've been running the workflow with so far:
- NT run
- 2030 planning horizon

**Instructions:**
1. Open `config/config.tyndp.yaml` in a text editor
2. Set `run: name:` to `"NT"`. The top section of your file should look like:

```yaml
run:
  prefix: "tyndp"
  name: "NT"
```

3. Define `scenario: planning_horizons:` to `2030`, so that the scenario section should look like this:

```yaml
scenario:
  clusters:
  - all
  planning_horizons:
  - 2030
```

3. Save the file
4. Then run the dry-run: `! pixi run tyndp-sb -n`

You should see the same number of steps as when we ran `! pixi run tyndp-sb --config 'run={"name":"NT"}' 'scenario={"planning_horizons":[2030]}' -n` above.

Now you can run the workflow without needing to specify configuration overrides on the command line!

In [ ]:
! pixi run tyndp-sb -n

Optional: We can now actually run the workflow once on a simplified temporal resolution (168H) which is computationally feasible to solve within this workshop. This resolution is already pre-configured, so we can simply execute:

```bash
! pixi run tyndp-sb
```

If you are running this notebook locally or executing the workflow directly from a terminal, the workflow will automatically launch a dashboard called PyPSA-Explorer in your default browser at http://localhost:8060. We will take some time later on to dive into this Explorer, but feel free to explore it already when you get to this point.

:::{warning}

The first execution of the workflow can take varying between 15-60 minutes depending e.g. on your internet connection or the computational resources of your machine. Especially, when executing the workflow via this notebook on Google Colab, this can take a while the first time you run it.

:::

If executing the workflow is taking too long, you can alternative run the following cell to skip the initial execution of the workflow and directly place the generated files into the data, resources and results directories.

You will then be able to skip most if not all of the workflow.

**Note:** If you are running into issues with this approach, you can try adding the following option which will tell Snakemake to only account for the timestamps of files when determining which steps to re-execute.

```bash
pixi run tyndp-sb --rerun-trigers mtime
```

In [ ]:
# comment in the data entry in the following dictionary to retrieve the full input data for the workflow (~ 11GB)
urls = {
    # "data/data-workshop.zip": "https://storage.googleapis.com/open-tyndp-data-store/workshop-trondheim/data-workshop.zip",
    "resources/resources-workshop.zip": "https://storage.googleapis.com/open-tyndp-data-store/workshop-trondheim/resources-workshop.zip",
    "results/results-workshop.zip": "https://storage.googleapis.com/open-tyndp-data-store/workshop-trondheim/results-workshop.zip",
    ".snakemake.zip": "https://storage.googleapis.com/open-tyndp-data-store/workshop-trondheim/.snakemake.zip",
}

if os.path.basename(os.getcwd()) in ["open-tyndp"]:
    for name, url in urls.items():
        if os.path.exists(name):
            print(f"File {name} already exists. Skipping download.")
        else:
            print(f"Retrieving {name} from storage.")
            urlretrieve(url, name)
            print(f"File available in {name}.")

        print(f"Unzipping and overwriting {name.split("/")[0].removesuffix(".zip")}.")
        try:
            unzip_with_timestamps(name, name.split("/")[0].removesuffix(".zip"))
        except zipfile.BadZipFile:
            print(f"{name} is corrupt, deleting. Re-run this cell to re-download.")
            os.remove(name)
        print("Done.")
else:
    print("Not in open-tyndp directory.")

In [ ]:
# ! pixi run tyndp-sb --rerun-triggers mtime

Let's save the outputs into a dictionary for later inspection.

In [ ]:
# Define the path and an importer function
base_path = "results/tyndp/NT/networks/"


def import_network(fn: str):
    n = pypsa.Network(fn)
    n.sanitize()
    return n


# Load the latest NT 2030 scenario network directly into dictionary for PyPSA-Explorer
networks = {
    "NT 2030": import_network(base_path + "base_s_all___2030.nc"),
}

## Modifying assumptions in Open-TYNDP

A key advantage of open-source energy modelling is the ability to adjust input assumptions and observe how results respond. This is especially important for TYNDP models, where assumptions evolve over time and multiple configurations need testing.
Open-TYNDP offers several ways to do this, ranging from simple data updates to custom constraints:

| Method | What it does | Where |
|---|---|---|
| **Input data** | Replace or update raw input files | `data/` folder |
| **Custom assumptions** | Override cost & technology parameters for specific technologies and planning horizons | `data/custom_cost.csv` |
| **Adjustments** | Apply scaling factors or absolute overrides to specific components directly in the scenario config | `config/scenarios.tyndp.yaml` |
| **Custom constraints** | Add or modify optimisation constraints beyond what the config exposes | `scripts/solve_network.py` |

Methods are ordered from simplest to most advanced — for most use cases, input data or custom assumptions are the right starting point.

### Method 1: Modifying Input Data

The most direct method to modify the model is by editing the input data files. We've already retrieved a prebuilt version of the open-tyndp GitHub repository into our working directory dated to the 29th of November 2025.

Navigate to the `data` directory in the `open-tyndp` repository and locate the `tyndp_2024_bundle` folder:

In [ ]:
from scripts._helpers import display_tree

target_directory = "data/tyndp/archive/2024"
print("data")
display_tree(target_directory, max_depth=1)

You can browse and replace any input files with your own data, provided they follow the **same format and structure** as the original files.

### Method 2: Custom Assumptions

To override specific assumptions—such as capital costs, marginal costs, or technical parameters for particular technologies—use a long-format CSV file called `custom_cost.csv` in the `data` folder.

Open `custom_cost.csv` to see example entries that illustrate the required structure:

In [ ]:
custom_cost = pd.read_csv("data/custom_costs.csv")
custom_cost

To add your own custom assumptions, insert a new row in this CSV file. At minimum, specify the `planning_horizon`, `technology`, `parameter`, `value`, and `unit` for each entry. As shown above, you can also use the `all` keyword to override a value for all technologies and/or all planning horizons simultaneously.

These custom entries are automatically applied when you run the Open-TYNDP workflow.

### Method 3: Adjustments

The third method involves making targeted changes to specific technologies or components directly in the model configuration.

These are called `adjustments`, and you define them in the scenario configuration file.

Here's an example from the default configuration file:

In [ ]:
display_code_lines("config/config.default.yaml", "yaml", 1225, 1233)

As shown, you can specify either a scaling factor or an absolute value for any combination of:
- **Component type** (e.g., Generator, Link, Load)
- **Carrier** (i.e., technology type)
- **Attribute** (e.g., marginal_cost, efficiency, p_nom)

To apply manual adjustments to a scenario you're modeling, add an `adjustments` section to your `scenarios.tyndp.yaml` file.

Let's examine the structure of that file:

In [ ]:
display_code_lines("config/scenarios.tyndp.yaml", "yaml", 1, 50)

### Method 4: Custom Constraints

You may also want to add custom constraints to the model. Since we haven't covered PyPSA constraints in detail yet, we won't dive deep here—we'll cover them comprehensively in a future workshop. For now, it's useful to know where you would add a custom constraint that isn't already represented through existing component parameters. For more information, see the [PyPSA documentation on custom constraints](https://docs.pypsa.org/latest/user-guide/optimization/custom-constraints/).

**Note**: Many constraints are already built into PyPSA. For example, to add an upper limit on the expansion capacity of a Link, you can simply use the `p_nom_max` attribute. PyPSA automatically translates this into a binding upper-limit constraint during optimization.

For custom constraints that go beyond existing parameters, you can insert your own code directly into the model. This flexibility is one of the key advantages of working with an open-source framework.

Navigate to the `scripts` directory of the `open-tyndp` repository and open the `solve_network.py` script. The following are two example functions for defining such custom constraints:

In [ ]:
display_code_lines("scripts/solve_network.py", "Python", 1320, 1400)

In `solve_network.py`, you'll find a function called `extra_functionality`. This is where additional custom constraints are added to the optimization model before solving.

For instance, Open-TYNDP includes a custom constraint for offshore hubs by calling the nested function `add_offshore_hubs_constraint`. This constraint limits the expansion of DC and H2 wind farms at the same location and enforces maximum potential per zone according to zone trajectories.

You can add your own constraints in the same location. Browse the existing constraint implementations to understand the structure and coding style.

We'll cover PyPSA constraint mechanics in depth in a future workshop.

### Task: Apply Manual Adjustments

Open `data/open-tyndp/config/scenarios.tyndp.yaml` and update the `NT` scenario with the following manual adjustments:

- Increase the `marginal_cost` of all H2 imports by a factor of 1.5 (2030) and 1.3 (2040). The supply of imported H2 is included as a `Generator` component with the carrier name `import H2`.
- Change the `efficiency` of `H2 Electrolysis` to 78% for both 2030 and 2040. H2 Electrolysis is added as a `Link` component.
- Remove the initial capacity (`p_nom` and `p_nom_min`) of all `solar-pv-utility` generators for 2030.

**Optional**: Now run the model (just for 2030) and explore the results in the model dashboard `PyPSA-Explorer`. If you are executing this locally, the Explorer will automatically launch. For this simply execute again:

```bash
pixi run tyndp-sb
```

**Hint I**: Always start with a dry run first (add `-n` to your Snakemake command).

**Hint II**: Make sure that the scenario that you want to run is specifying a solver that you can use. You can change the solver to Highs (an open-source solver) by changing the following configuration: `solving:solver`.

**Hint III**: If you are running into issues with executing the workflow after retrieving the checkpoint data (only about 30 rules should be re-triggered), you can try adding the following option which will tell Snakemake to only account for the timestamps of files and changed config parameters when determining which steps to re-execute.

```bash
pixi run tyndp-sb --rerun-trigers mtime params
```

# 3. Exploring Open-TYNDP's latest outcomes

Now that you are familiar with the model workflow and how to make easy adjustments to the model, let's have a look at the latest Open-TYNDP NT scenario results (v0.7.1) and explore them interactively using the model dashboard PyPSA-Explorer.

## Interactive Exploration with PyPSA-Explorer

PyPSA-Explorer is an interactive dashboard for visualizing and analyzing energy system networks. It provides:

**1. Energy Balance Tab**
   - View production, consumption, and storage patterns over time
   - Switch between time series and aggregated views
   - Filter by energy carrier (electricity, hydrogen, etc.)
   - Filter by country or region

**2. Capacity Tab**
   - Analyze installed capacities across scenarios
   - Compare capacity buildout between 2030 and 2040
   - View breakdowns by technology type and region

**3. Economics Tab**
   - Examine costs and revenues
   - Review CAPEX and OPEX breakdowns by technology
   - Compare regional cost distributions
   - Assess investment requirements

**4. Network Map**
   - Visualize the geographical network layout
   - View an interactive map with network components
   - Zoom and pan to explore specific regions

**Tip:** Use the scenario selector buttons in the top-right corner to switch between NT 2030 and NT 2040 scenarios.

:::{note}

 PyPSA-Explorer can be launched in different ways depending on your environment:

- **Local Jupyter**: Use the terminal command (recommended) or inline display
- **Google Colab**: The dashboard launches inline, embedded directly in the notebook
- **Local Open-TYNDP**: Launch the PyPSA-Explorer directly from the Open-TYNDP either directly from the workflow or from archived, pre-solved networks

Follow the instructions below for your specific environment.

:::

:::{warning}

 PyPSA-Explorer is under active development and can experience bugs and limitations.

:::

#### For Local Users

If you're running locally, we **recommend** launching PyPSA-Explorer directly from the Open-TYNDP workflow using:

```bash
pixi run tyndp-sb
```

Or from the terminal directly using the pypsa-explorer pip package:

```bash
pypsa-explorer path/to/your/network.nc:network_name path/to/your/second/network.nc:network2_name
```

This command opens the dashboard in your default browser at http://localhost:8050.

**Alternative**: The cell below can launch the dashboard inline within the notebook, though the terminal method provides better performance and responsiveness.

```python
# Terminal method recommended
USE_TERMINAL = True  # Change to False if you want to launch from the notebook instead

if not IN_COLAB and not USE_TERMINAL:
    # Local Jupyter: Inline display
    app = create_app(networks)  # Make sure to define `networks` as a dictionary of imported PyPSA Networks first
    app.run(jupyter_mode="tab", port=port, debug=False)
```

#### For Google Colab Users

Running PyPSA-Explorer on Google Colab requires a small workaround to display the dashboard properly inside the notebook.

We already imported a useful helper function to handle this: `run_pypsa_explorer_in_colab()`

```python
if IN_COLAB:
    run_pypsa_explorer_in_colab(networks, port)
```

**Tip for Colab users:** To view the dashboard in fullscreen mode, click the three dots (⋮) in the top-right corner of the output cell and select **"View output fullscreen"**.

## Task: Explore outcomes with ``PyPSA-Explorer``

Try to find and verify some specific information about the solved networks.

(a) Can you verify the total amount of wind generated on Danish Offshore Hubs in 2040 at *43 TWh*?

(b) Can you verify Germany's *net annual imports of H2* in 2040 at *314 TWh*? How does it compare to other countries?

**Hint:** Look for `H2 pipeline`, `H2 Import Pipeline` and `H2 Import LH2` in the energy balance.

(c) Can you verify the observed correlation between electricity mix and H2 production in Germany for the *first week of June* in 2040? What can we notice?

Let's load the latest Open-TYNDP NT scenario outcomes (v0.7.1) that we retrieved in the beginning.

In [ ]:
# Define the path and an importer function
base_path = "../results-0.7.1/NT-cy2009-20260520/networks/"


def import_network(fn: str):
    n = pypsa.Network(fn)
    n.sanitize()
    return n


# Load the latest NT scenario networks directly into dictionary for PyPSA-Explorer
networks_071 = {
    "NT 2030": import_network(base_path + "base_s_all___2030.nc"),
    "NT 2040": import_network(base_path + "base_s_all___2040.nc"),
}

You can now launch the PyPSA-Explorer with these networks either locally or from within this Colab notebook as we showed you before.

Alternatively, if you are running commands directly in your terminal or PowerShell, you can make use of a handy pixi task that we've defined for investigating pre-solved Open-TYNDP network:

```bash
! pixi run launch-presolved-explorer
```

In [ ]:
# Terminal method recommended
USE_TERMINAL = True  # Change to False if you want to launch from the notebook instead

if not IN_COLAB and not USE_TERMINAL:
    # Local Jupyter: Inline display
    app = create_app(
        networks_071
    )  # Make sure to define `networks` as a dictionary of imported PyPSA Networks first
    app.run(jupyter_mode="tab", port=port, debug=False)

In [ ]:
# Set port for explorer
port = 8070

if IN_COLAB:
    run_pypsa_explorer_in_colab(networks_071, port)

If you are launching the explorer from the workflow, you will need to update the configuration file again to include both 2030 and 2040 planning years:

```yaml
scenario:
  clusters:
  - all
  planning_horizons:
  - 2030
  - 2040
```

In [ ]:
# ! pixi run launch-presolved-explorer

## (Optional) Explore NT outcomes using the ``PyPSA.statistics`` module

When Open-TYNDP runs an optimisation, all input data, installed capacities, demand profiles, network topology, technology assumptions, is loaded into the PyPSA network object `n`. After the optimisation completes, the results are stored in the same network object. This means `n` contains everything: what went in and what came out.

Since a PyPSA network holds a large amount of detailed information across many components, exploring it directly can be overwhelming. We already saw how the `PyPSA-Explorer` Dashboard can make this a lot easier. But there is also a more hands-on approach that gives you more freedom to extract the exact metrics that you need while handling a lot of the overhead of processing the raw model data. 

This is where `n.statistics` comes in: a built-in module that gives you fast, easy access to the most important system-level metrics without having to dig into the raw network data yourself.

`n.statistics` provides a consistent, high-level API that handles component iteration, port mapping, and carrier grouping automatically.

:::{tip}

`n.stats` is available as a shorthand alias for `n.statistics`.

:::

Each method can be called individually or explored via the summary table:

| Category | Methods |
|---|---|
| **Costs** | `capex()`, `installed_capex()`, `expanded_capex()`, `opex()`, `system_cost()` |
| **Capacity** | `installed_capacity()`, `optimal_capacity()`, `expanded_capacity()`, `capacity_factor()` |
| **Energy** | `supply()`, `withdrawal()`, `energy_balance()`, `transmission()`, `curtailment()` |
| **Market** | `prices()`, `revenue()`, `market_value()` |

Every method accepts the same filtering and grouping parameters:
| Parameter | Description |
|---|---|
| `groupby` | String, list, or callable — how to group results (default: `\"carrier\"`) |
| `groupby_method` | Aggregation function (`\"sum\"` (default), `\"mean\"`, …) |
| `groupby_time` | `\"sum\"`, `\"mean\"`, or `False` for time series — default varies by method |
| `components` | Filter to specific component types |
| `carrier` | Filter by carrier name (internal name) |
| `bus_carrier` | Filter by the carrier of the bus |
| `nice_names` | Use human-readable carrier names (default: `True`) |

:::{warning}

`prices()` has a simplified interface — `groupby` and `groupby_time` are booleans,
and it does not accept `carrier` or `components`.

:::

The full `PyPSA.statistics` API documentation is available in the [pypsa documentation](https://docs.pypsa.org/latest/user-guide/statistics/). Additionally, you can find two video tutorials on *PyPSA meets Earth*'s youtube channel ([part 1](https://www.youtube.com/watch?v=_Asjhz6We5o), [part 2](https://www.youtube.com/watch?v=nlh3azf2JJw)) for more comprehensive information and examples on how to use the statistics module. This learning material is open-source and available on [GitHub](https://github.com/open-energy-transition/pypsa-training-sessions).

### Extracting insights from a network

To start of we'll have a look at the network for the `NT 2030` scenario.

In [ ]:
scenario = "NT 2030"


To avoid typing `networks_071["NT 2030"].stats` every time, let's save a shortcut:


In [ ]:
s2030 = networks_071["NT 2030"].stats
s2040 = networks_071["NT 2040"].stats

You can easily get a comprehensive overview of all system-level metrics at once.

In [ ]:
s2030()

Of course, this can be a bit difficult to grasp. So let's have a look at some specific outputs instead.

We can investigate electricity supply and demand for our NT 2030 network using the `energy_balance` method:

In [ ]:
balance = (
    s2030.energy_balance(
        bus_carrier=["AC", "low voltage"],
        groupby=["carrier"],
        aggregate_across_components=True,
    ).div(
        1e6
    )  # TWh
    # .sort_values(ascending=False)
    .sort_index(ascending=False)
)

# Format output
balance = balance[
    abs(balance.values) > clip
].to_frame(  # Filter for entries > clipped value
    "Supply (+), Demand (-) [TWh]"
)
balance.style.format("{:,.2f}")  # Make style a bit prettier

### Compare results with ``PyPSA.NetworkCollections``

PyPSA v1.0 introduced a new object called `NetworkCollection` that lets you query multiple pypsa networks at once, so you can compare planning years side by side without repeating your code.



Let's have a look at how this is used in practice. First we define a network collection (`nc`) with our previously imported result networks for NT 2030 and NT 2040.

In [ ]:
nc = pypsa.NetworkCollection(networks_071)
nc

As we can see, our `NetworkCollection` contains two networks, the `NT 2030` network and the `NT 2040` network.

We can now use `PyPSA.statistics` accessor directly on this `NetworkCollection` instead of a single network to get the metrics for them simultaneously.

Let's start by defining a helper variable `sc` for the statistics accessor to make our life a bit easier going forward.

In [ ]:
sc = nc.statistics

With this, we can extract electricity prices in the system across NT planning years. In line with the Market Model, we will aggregate the outputs using an average (`weighting="time"`).

Note that you can easily choose if you want to calculate the price weighted by load instead by selecting `weighting='load'`.

In [ ]:
prices = sc.prices(bus_carrier="AC", weighting="time").unstack("network")
prices.head(10)

For easier readability, we can plot them:

In [ ]:
prices.plot.bar(
    figsize=(25, 4),
    edgecolor="white",
    ylabel="€/MWh",
    xlabel="Bus Carrier",
    title="Electricity Price by node",
);

:::{Note}

Please keep in mind that the methodology used to implement hydrogen and electricity market coupling slightly differs from the TYNDP 2024 approach. The official TYNDP process uses a proprietary market model for the least-cost optimisation. Unlike this market model, which assumes a fixed hydrogen fuel price for hydrogen-to-power generation, Open-TYNDP, built on the open-source PyPSA/PyPSA-Eur framework, couples electricity and hydrogen markets by using an endogenous hydrogen fuel price. This means hydrogen prices are determined within the optimisation rather than set externally.

One consequence of this coupling is that price spikes can occur when the model resorts to load shedding. That is, when demand cannot be fully met and the model artificially sheds load at a very high cost to balance the system. Load shedding is a standard technique in energy system modelling to prevent the optimisation from becoming infeasible. It acts as a last resort that allows the model to always find a solution, at the cost of unmet demand. These spikes are especially visible in the NT 2040 electricity price outcomes. Detailed electricity price benchmarks excluding load shedding are available [Zenodo](https://zenodo.org/records/20303009).

:::


### `PyPSA.Statistics` plotting APIs

As we already introduced in a previous workshop, there is actually a quick way to explore the data with plots generated directly using the `PyPSA.statistics` module.

We can now explore the electricity energy balance for the NT 2030 network using this API directly.

In [ ]:
fig, ax, _ = s2030.energy_balance.plot.bar(
    bus_carrier=["AC", "low voltage"],
    query=f"abs(value)>{clip * 1e6}",  # Values are in MWh
    height=6,
)

ax.set_title(f"Electricity Energy Balance {scenario} (Clipped at {clip} TWh)");

...or we can even interactively explore the production of a specific technology in a specific country. For example January wind production in the Netherlands for NT 2030:

In [ ]:
fig = s2030.energy_balance.iplot.area(
    facet_col="country",
    y="value",
    x="snapshot",
    carrier="wind",
    color="carrier",
    query="country == 'NL' and snapshot < '2009-02'",
    width=1200,
    height=500,
    title="Wind Production Netherlands NT 2030, January",
)

fig.update_layout(yaxis_title="Wind Production [MWh_el]")

As you can see, in January the Netherland's wind mix is largely dominated by Offshore Wind production.

And of course `NetworkCollections` also work with `PyPSA.statistics` quick plotting API. So we can have a look at the previous price plot but using statistics plotting directly:

In [ ]:
fig = sc.prices.iplot.bar(
    bus_carrier=["AC"],
    x="name",
    y="value",
    color="network",
    stacked=True,
    width=1200,
    height=500,
    nice_names=False,
    title="Electricity Price by node",
)

fig.update_layout(yaxis_title="Prices [EUR/MWh]")

## Task: Grow comfortable with ``PyPSA.statistics``
Familiarize yourself with the statistics module (again) and explore the latest outcomes of Open-TYNDP using the different methods and plots introduced today.

**Hint**: You can also refer to the [introduction](#reminder-the-pypsa-statistics-module) above for more information on the different methods and parameters of ``PyPSA.statistics``.

In [ ]:
# Your playground

## Task: Reproduce Benchmarks
(a) If you feel comfortable using `PyPSA.statistics`, you can try to reproduce the Open-TYNDP outcomes from the following example of our latest [benchmarking figures](https://zenodo.org/records/20303009).

Try it without looking at the previous example first.

In [ ]:
show_benchmarks(
    "benchmark_hydrogen_price_cy2009",
    [2030],
    "../results-0.7.1/NT-cy2009-20260520/benchmarks/tyndp-2024/graphics_s_all___all_years/by_bus",
)

(b) Advanced: Try to exclude load shedding from the hydrogen price in 2040.

**Hint:** The load shedding price is set as *3000 EUR/MWh_H2*.

In [ ]:
# Your solution a)

In [ ]:
# Your solution b)

## Task: Verify previous findings

(a) Can you verify the total amount of wind generated on Danish Offshore Hubs in 2040 at *43 TWh*?

**Hint:** The Offshore Hub bus carrier is `AC_OH`. Remember to include the Bornholm Energy Island bus called `BEIOH01`.

(b) Can you verify Germany's *net annual imports of H2* in 2040 at *314 TWh*? How does it compare to other countries?

**Hint:** Look for `carrier="H2 pipeline|import"` in the energy balance. Remember that you can group by `bus` or `country`.

(c) Can you investigate the correlation between electricity mix and H2 production in Germany for the *first week of June* in 2040? What can we notice?

In [ ]:
# Your solution a)

In [ ]:
# Your solution b)

In [ ]:
# Your solution c)

# Solutions

## Task: Apply Manual Adjustments

Add the following `adjustments` section for the `NT` scenario configuration in `scenarios.tyndp.yaml`:

```yaml
adjustments:
  sector:
    factor:
      Generator:
        import H2:
          marginal_cost:
            2030: 1.5
            2040: 1.3
        solar-pv-utility:
          p_nom:
            2030: 0.0
    absolute:
      Link:
        H2 Electrolysis:
          efficiency:
            2030: 0.78
            2040: 0.78
```

In a terminal window, we start with a dry run to preview which rules will execute:

```bash
! pixi run tyndp-sb -n
```

The dry run shows that, because this scenario was run previously *without* the adjustments, Snakemake will only re-execute the rules affected by your manual changes.

Now run the workflow for real:

```bash
! pixi run tyndp-sb
```

In [ ]:
# ! pixi run tyndp-sb

If running the workflow from within this notebook or on Colab, we need to manually launch PyPSA-Explorer again to inspect the results of our modified scenario.

First, we load the new outputs into a new `networks_new` dictionary:

In [ ]:
# Update networks dictionary with new 2030 network for PyPSA-Explorer
# base_path = "results/tyndp/NT/networks/"
#
# def import_network(fn: str):
#     n = pypsa.Network(fn)
#     n.sanitize()
#     return n
#
# networks_new = {
#     "NT 2030 new": import_network(base_path + "base_s_all___2030.nc"),
# }
#
# # set a new port for the updated dashboard
# port = 8070
#
# networks_new

In [ ]:
# Terminal method recommended
# USE_TERMINAL = False  # Change to False if you want to launch inline display
#
# if not IN_COLAB and not USE_TERMINAL:
#     # Local Jupyter: Inline display
#     app = create_app(networks_new)
#     app.run(jupyter_mode="tab", port=port, debug=False)

In [ ]:
# if IN_COLAB:
#     run_pypsa_explorer_in_colab(networks_new, port)

## Task: Reproduce benchmarks

In [ ]:
# (a) Try to reproduce the Open-TYNDP outcomes from the hydrogen prices example above from our latest [benchmarking figures](https://zenodo.org/records/20303009).

show_benchmarks(
    "benchmark_hydrogen_price_cy2009",
    [2030],
    "../results-0.7.1/NT-cy2009-20260520/benchmarks/tyndp-2024/graphics_s_all___all_years/by_bus",
)

In [ ]:
prices = (
    s2030.prices(bus_carrier="H2", weighting="time").to_frame("Open-TYNDP").sort_index()
)

ax = prices.plot.bar(figsize=(16, 4), ylabel="EUR/MWh_H2", xlabel="Node")

ax.set_facecolor("#e8e8e8")
ax.set_axisbelow(True)
ax.grid(True)

# Legend with average
handles, labels = ax.get_legend_handles_labels()
new_labels = [f"{label} ({prices[label].mean():.1f} EUR/MWh_H2)" for label in labels]
ax.legend(handles, new_labels, loc="upper left")

ax.set_title("Hydrogen Price - Scenario NT - CY 2009 - Year 2030")
ax.tick_params(axis="x", rotation=45)

In [ ]:
# (b) Optional: Try to exclude load shedding from the hydrogen price in 2040.
voll = 3000  # EUR/ MWh_H2
prices = (
    s2040.prices(
        bus_carrier="H2",
        weighting="time",
        groupby_time=False,
    )
    .pipe(lambda x: x.where(x < voll * 0.98))  # Add 2% of numerical tolerance
    .mean(axis=1)
    .to_frame("value")
    .sort_index()
)

ax = prices.plot.bar(figsize=(16, 4), ylabel="EUR/MWh_H2", xlabel="Node")

ax.set_facecolor("#e8e8e8")
ax.set_axisbelow(True)
ax.grid(True)

# Legend with average
handles, labels = ax.get_legend_handles_labels()
new_labels = [f"{label} ({prices[label].mean():.1f} EUR/MWh_H2)" for label in labels]
ax.legend(handles, new_labels, loc="upper left")

ax.set_title("Hydrogen Price excl. Load Shedding - Scenario NT - CY 2009 - Year 2040")
ax.tick_params(axis="x", rotation=45)

## Task: Investigate outputs

In [ ]:
# (a) Can you verify the total amount of wind generated on Danish Offshore Hubs in 2040 at *43 TWh*?
(
    s2040.energy_balance(
        bus_carrier="AC_OH",
        aggregate_across_components=True,
        groupby=["bus", "carrier"],
        components="Generator",
    )
    .div(1e6)  # TWh
    .to_frame("Wind Generation [TWh]")
    .loc[["BEIOH01", "DKWOH01"]]
    .sum()
)

In [ ]:
# (b) Can you verify Germany's *net annual imports of H2* in 2040 at 314 TWh? How does it compare to other countries?
(
    s2040.energy_balance(
        bus_carrier="H2",
        carrier="H2 pipeline|import",
        aggregate_across_components=True,
        groupby=["carrier", "country"],
    )
    .div(1e6)  # TWh
    .droplevel("carrier")
    .groupby("country")
    .sum()
    .sort_values(ascending=False)
    .to_frame("H2 Import (+), H2 Export (-) [TWh]")
    .head()
    .style.format("{:,.2f}")  # Make style a bit prettier
)

In [ ]:
# (c) Can you investigate the correlation between electricity mix and H2 production in Germany for the *first week of June* in 2040? What can we notice?
s2040.energy_balance.iplot.area(
    bus_carrier=["AC", "H2"],
    y="value",
    x="snapshot",
    color="carrier",
    stacked=True,
    facet_row="bus_carrier",
    facet_col="country",
    sharex=False,
    sharey=False,
    query="snapshot >= '2009-06-01' and snapshot < '2009-06-08' and country == 'DE'",
    width=1200,
    height=500,
)

We can observe for night of the 5th of June, that wind production drops along with solar electricity production resulting in no hydrogen production via electrolysis for that time. Instead, German hydrogen demand is met via H2 pipeline imports as well as from Cavern Storages and blue Hydrogen production. Pumped hydro, battery storage and electricity imports are utilized to support electricity production in the same period to meet the remaining exogenous electricity demand.

# Notebook clean up

In [ ]:
# Only clean up data when running in CI environment
if os.getenv("CI"):
    rm_dir = "data/results-0.7.1"
    print(
        f"CI environment detected. Cleaning up notebook data by removing '{rm_dir}' and '{rm_dir}.zip'."
    )
    shutil.rmtree(rm_dir, ignore_errors=True)
    Path(f"{rm_dir}.zip").unlink(missing_ok=True)
else:
    print("Skipping cleanup (not in CI environment).")